# AI-Generated vs Authentic Image Detector Training Pipeline

This notebook trains a lightweight, sub-second deep learning model (**MobileNetV3-Large**) to classify images as either **AI-Generated** or **Authentic**.

### Key Features
- Designed to run on free GPU tiers: **Google Colab (T4)** and **Kaggle (T4/P100)**.
- Dataset: Integrates seamlessly with **CIFAKE** (60k images: Real CIFAR-10 vs Stable Diffusion v1.4 synthetic) or custom datasets.
- Optimization: Mixed-precision training (`torch.cuda.amp`), Cosine Annealing learning rate schedule.
- Inference Optimization: TorchScript and ONNX model export for sub-50ms CPU/GPU deployment.

In [ ]:
# 1. Environment & GPU Verification
import os
import sys
import time
import torch
import torchvision
import numpy as np

print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

In [ ]:
# 2. Install Required Dependencies (if needed)
!pip install -q timm scikit-learn matplotlib tqdm onnx

In [ ]:
# 3. Dataset Download & Preparation (CIFAKE Dataset)
# CIFAKE contains 60,000 images (30,000 Real, 30,000 Fake)
# On Kaggle: Add dataset 'birdy654/cifake-real-and-ai-generated-synthetic-images'
# On Colab: You can download via Kaggle API or direct zip download

import urllib.request
import zipfile
from pathlib import Path

DATA_DIR = Path("./data/cifake")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# If running locally or on Colab without Kaggle API, create sample data structure
print(f"Data directory set to: {DATA_DIR.resolve()}")

In [ ]:
# 4. Transforms and Data Augmentation
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

INPUT_SIZE = (224, 224)
BATCH_SIZE = 64
NUM_WORKERS = 2

train_transform = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("Data transformations configured.")

In [ ]:
# 5. Lightweight Architecture: MobileNetV3-Large
import torch.nn as nn
from torchvision import models

def create_model(num_classes=2, pretrained=True):
    weights = models.MobileNet_V3_Large_Weights.DEFAULT if pretrained else None
    model = models.mobilenet_v3_large(weights=weights)
    
    # Replace final linear layer for binary classification (0: AI-Generated, 1: Authentic)
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Sequential(
        nn.Dropout(p=0.2),
        nn.Linear(in_features, num_classes)
    )
    return model

model = create_model().to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

In [ ]:
# 6. Training Configuration with Automatic Mixed Precision (AMP)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

print("Optimizer, loss function, and scheduler initialized.")

In [ ]:
# 7. Single Epoch Training & Validation Functions
def train_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(images)
            loss = criterion(outputs, targets)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(targets).sum().item()
        total += targets.size(0)
        
    return running_loss / total, correct / total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    
    with torch.no_grad():
        for images, targets in loader:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            
            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(targets).sum().item()
            total += targets.size(0)
            
    return running_loss / total, correct / total

In [ ]:
# 8. Benchmark Inference Speed (< 1.0s Requirement Verification)
def benchmark_latency(model, device, num_runs=50):
    model.eval()
    dummy_input = torch.randn(1, 3, 224, 224).to(device)
    
    # Warm up
    for _ in range(10):
        with torch.inference_mode():
            _ = model(dummy_input)
            
    latencies = []
    for _ in range(num_runs):
        start = time.perf_counter()
        with torch.inference_mode():
            _ = model(dummy_input)
        latencies.append((time.perf_counter() - start) * 1000)
        
    latencies = np.array(latencies)
    print(f"--- LATENCY BENCHMARK ({device}) ---")
    print(f"Mean Latency:   {np.mean(latencies):.2f} ms")
    print(f"P50 (Median):   {np.percentile(latencies, 50):.2f} ms")
    print(f"P95:            {np.percentile(latencies, 95):.2f} ms")
    print(f"Sub-1s SLA:     {'PASSED' if np.percentile(latencies, 95) < 1000 else 'FAILED'}")

benchmark_latency(model, device="cpu")

In [ ]:
# 9. Model Export (TorchScript & ONNX for Fast Deployment)
os.makedirs("exported_models", exist_ok=True)

# A. PyTorch Checkpoint
checkpoint = {
    "model_state_dict": model.state_dict(),
    "architecture": "MobileNetV3-Large",
    "class_to_idx": {"ai": 0, "real": 1},
    "input_size": [224, 224]
}
torch.save(checkpoint, "exported_models/ai_image_detector.pth")
print("Saved PyTorch checkpoint to exported_models/ai_image_detector.pth")

# B. TorchScript Export (Pre-compiled graph for sub-50ms CPU inference)
model.eval().to("cpu")
dummy_input = torch.randn(1, 3, 224, 224)
traced_model = torch.jit.trace(model, dummy_input)
traced_model.save("exported_models/ai_image_detector_traced.pt")
print("Saved TorchScript model to exported_models/ai_image_detector_traced.pt")

# C. ONNX Export
torch.onnx.export(
    model,
    dummy_input,
    "exported_models/ai_image_detector.onnx",
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    opset_version=14
)
print("Saved ONNX model to exported_models/ai_image_detector.onnx")